# ducklake-dataframe Pandas Tutorial

This notebook walks through all the main features of `ducklake-dataframe` using the **Pandas** wrapper (`ducklake_pandas`).

It mirrors the [Polars tutorial](tutorial.ipynb) so you can compare the two APIs side-by-side.

In [ ]:
# Install/upgrade to latest versions
!pip install -U ducklake-dataframe[pandas]
!pip install duckdb==1.5.2

In [ ]:
import os
import shutil
import tempfile

import duckdb
import pandas as pd
import numpy as np

# Work directory
WORKDIR = tempfile.mkdtemp(prefix="ducklake_pandas_tutorial_")
CATALOG = os.path.join(WORKDIR, "catalog.ducklake")
DATAPATH = os.path.join(WORKDIR, "data/")
print(f"Working directory: {WORKDIR}")

## 1. Initialize the Catalog with DuckDB

DuckLake catalogs must be initialized using DuckDB.
After that, `ducklake_pandas` can read/write independently.

In [ ]:
con = duckdb.connect()
con.execute("INSTALL ducklake; LOAD ducklake")
con.execute(
    f"ATTACH 'ducklake:sqlite:{CATALOG}' AS lake "
    f"(DATA_PATH '{DATAPATH}')"
)
con.execute("""
    CREATE TABLE lake.users (
        id     INTEGER,
        name   VARCHAR,
        region VARCHAR
    )
""")
con.execute("""
    INSERT INTO lake.users VALUES
        (1, 'Alice',   'us'),
        (2, 'Bob',     'eu'),
        (3, 'Charlie', 'us'),
        (4, 'Diana',   'eu'),
        (5, 'Eve',     'us')
""")

In [ ]:
# Also create an events table for later
con.execute("""
    CREATE TABLE lake.events (
        ts      TIMESTAMP,
        user_id INTEGER,
        action  VARCHAR,
        region  VARCHAR
    )
""")
con.execute("""
    INSERT INTO lake.events VALUES
        ('2025-03-01 10:00:00', 1, 'login',  'us'),
        ('2025-03-01 10:05:00', 2, 'login',  'eu'),
        ('2025-03-01 10:10:00', 1, 'click',  'us'),
        ('2025-03-01 10:15:00', 3, 'login',  'us'),
        ('2025-03-01 10:20:00', 2, 'click',  'eu')
""")
con.close()
print("Catalog created with 'users' (5 rows) and 'events' (5 rows)")

## 2. Reading with ducklake_pandas

No DuckDB needed — reads metadata from SQLite and data from Parquet directly.

> **Key difference from Polars:** Pandas has no lazy API, so there's no `scan_ducklake`.
> Use `read_ducklake` with `predicate` and `columns` for pushdown.

In [ ]:
from ducklake_pandas import read_ducklake

# Eager read
df = read_ducklake(CATALOG, "users")
print(f"Full table ({len(df)} rows):")
df

In [ ]:
# Predicate pushdown — lambda receives a DataFrame, returns a boolean Series
us_users = read_ducklake(
    CATALOG, "users",
    predicate=lambda d: d["region"] == "us",
)
print(f"US users only ({len(us_users)} rows):")
us_users

In [ ]:
# Column projection
names = read_ducklake(CATALOG, "users", columns=["id", "name"])
names

## 3. Catalog Inspection

The `DuckLakeCatalog` class provides Python equivalents of DuckLake's catalog functions.

In [ ]:
from ducklake_pandas import DuckLakeCatalog

catalog = DuckLakeCatalog(CATALOG)
print(f"Schemas:   {catalog.list_schemas()['schema_name'].tolist()}")
print(f"Tables:    {catalog.list_tables()['table_name'].tolist()}")
print(f"Snapshots: {len(catalog.snapshots())} total")
print(f"Current:   {catalog.current_snapshot()}")

In [ ]:
print("=== Table Info ===")
print(catalog.table_info())

print("\n=== Files in 'users' ===")
print(catalog.list_files("users"))

## 4. Writing with ducklake_pandas

### 4a. Append rows

In [ ]:
from ducklake_pandas import write_ducklake

new_users = pd.DataFrame({
    "id":     [6, 7],
    "name":   ["Frank", "Grace"],
    "region": ["eu", "us"],
})
write_ducklake(new_users, CATALOG, "users", mode="append")

print(f"After append: {len(read_ducklake(CATALOG, 'users'))} rows")
read_ducklake(CATALOG, "users")

### 4b. Verify from DuckDB

DuckDB can read what `ducklake_pandas` wrote — full interoperability.

In [ ]:
con = duckdb.connect()
con.execute("LOAD ducklake")
con.execute(f"ATTACH 'ducklake:sqlite:{CATALOG}' AS lake (DATA_PATH '{DATAPATH}')")
print(con.execute("SELECT * FROM lake.users ORDER BY id").df().to_string(index=False))

In [ ]:
con.close()  # release for ducklake_pandas

### 4c. Delete rows

In [ ]:
from ducklake_pandas import delete_ducklake

deleted = delete_ducklake(
    CATALOG, "users",
    predicate=lambda d: d["id"] == 2,
)
print(f"Deleted {deleted} row(s)")

### 4d. Update rows

In [ ]:
from ducklake_pandas import update_ducklake

updated = update_ducklake(
    CATALOG, "users",
    updates={"region": "apac"},
    predicate=lambda d: d["name"] == "Eve",
)
print(f"Updated {updated} row(s)")

### 4e. Merge (upsert)

> **Pandas predicates** use lambdas on DataFrames — not `pl.col()` expressions.

In [ ]:
from ducklake_pandas import merge_ducklake

source = pd.DataFrame({
    "id":     [1, 8],
    "name":   ["Alice2", "Hank"],
    "region": ["us", "eu"],
})
rows_upd, rows_ins = merge_ducklake(
    CATALOG, "users", source, on="id",
    when_matched_update=True,
    when_not_matched_insert=True,
)
print(f"Merge: {rows_upd} updated, {rows_ins} inserted")

### 4f. Verify from DuckDB again

In [ ]:
con = duckdb.connect()
con.execute("LOAD ducklake")
con.execute(f"ATTACH 'ducklake:sqlite:{CATALOG}' AS lake (DATA_PATH '{DATAPATH}')")
print(con.execute("SELECT * FROM lake.users ORDER BY id").df().to_string(index=False))

In [ ]:
con.close()

## 5. Time Travel

Every write creates a new snapshot. We can read the table at any historical snapshot.

In [ ]:
catalog = DuckLakeCatalog(CATALOG)
snapshots = catalog.snapshots()
print(f"Total snapshots: {len(snapshots)}")
snapshots[["snapshot_id", "snapshot_time"]]

In [ ]:
# Read at an earlier snapshot
snap_v = int(snapshots["snapshot_id"].iloc[4])
old_df = read_ducklake(CATALOG, "users", snapshot_version=snap_v)
print(f"At snapshot {snap_v}: {len(old_df)} rows")
old_df

## 5.5. Change Data Feed

`read_ducklake_changes` returns a row-level diff between two snapshot ids with a `change_type` column (`insert`, `delete`, `update_preimage`, `update_postimage`).


In [ ]:
from ducklake_pandas import read_ducklake_changes

snap_ids = sorted(catalog.snapshots()["snapshot_id"].tolist())
changes = read_ducklake_changes(
    CATALOG, "users", start_version=snap_ids[0], end_version=snap_ids[-1],
)
print(f"{len(changes)} changes between snapshots {snap_ids[0]} and {snap_ids[-1]}:")
print(changes.to_string(index=False))

# Just inserts / deletes via the catalog API
inserts = catalog.table_insertions("users", start_version=snap_ids[0], end_version=snap_ids[-1])
deletes = catalog.table_deletions("users", start_version=snap_ids[0], end_version=snap_ids[-1])
print(f"\nInsertions: {len(inserts)} rows; Deletions: {len(deletes)} rows")


## 6. Schema Evolution (DDL)

### Add, rename, drop columns, and change types

In [ ]:
from ducklake_pandas import (
    alter_ducklake_add_column,
    alter_ducklake_rename_column,
    alter_ducklake_drop_column,
    alter_ducklake_set_type,
)

# Add columns
alter_ducklake_add_column(CATALOG, "users", "email", "VARCHAR")
alter_ducklake_add_column(CATALOG, "users", "department", "VARCHAR")
print("Added 'email' and 'department' columns")

# Rename a column
alter_ducklake_rename_column(CATALOG, "users", "email", "contact")
print("Renamed email → contact")

# Drop a column
alter_ducklake_drop_column(CATALOG, "users", "department")
print("Dropped 'department'")

# Change type
alter_ducklake_set_type(CATALOG, "users", "id", "BIGINT")
print("Changed id: INTEGER → BIGINT")

In [ ]:
# Verify: DuckDB sees the schema changes
con = duckdb.connect()
con.execute("LOAD ducklake")
con.execute(f"ATTACH 'ducklake:sqlite:{CATALOG}' AS lake (DATA_PATH '{DATAPATH}')")
print(con.execute("DESCRIBE lake.users").df().to_string(index=False))

In [ ]:
con.close()

## 7. Partitioned Writes

Set partitioning on a table, then inserts automatically go into partition directories.

In [ ]:
from ducklake_pandas import alter_ducklake_set_partitioned_by

alter_ducklake_set_partitioned_by(CATALOG, "events", ["region"])

new_events = pd.DataFrame({
    "ts":      pd.to_datetime(["2025-03-02 09:00:00", "2025-03-02 09:05:00"]),
    "user_id": [6, 7],
    "action":  ["signup", "signup"],
    "region":  ["us", "eu"],
})
write_ducklake(new_events, CATALOG, "events", mode="append")
print("Appended partitioned events")

# Partition pruning — only reads 'us' partition files
us_events = read_ducklake(
    CATALOG, "events",
    predicate=lambda d: d["region"] == "us",
)
print(f"US events: {len(us_events)} rows (partition pruning)")

## 8. Data Inlining

Small inserts can be stored directly in the catalog database (SQLite) to avoid tiny Parquet files.

In [ ]:
from ducklake_pandas import create_ducklake_table

create_ducklake_table(CATALOG, "tiny", {"ts": "TIMESTAMP", "value": "DOUBLE"})

# Small insert → inlined in the catalog DB
small = pd.DataFrame({
    "ts":    pd.to_datetime(["2025-01-01 00:00:00"]),
    "value": [3.14],
})
write_ducklake(small, CATALOG, "tiny", mode="append", data_inlining_row_limit=100)
print(f"Inlined 1 row → {len(read_ducklake(CATALOG, 'tiny'))} rows total")

# Larger insert → Parquet
big = pd.DataFrame({
    "ts":    pd.to_datetime([f"2025-01-01 00:00:{i:02d}" for i in range(50)]),
    "value": [float(i) for i in range(50)],
})
write_ducklake(big, CATALOG, "tiny", mode="append")
print(f"After large append → {len(read_ducklake(CATALOG, 'tiny'))} rows total")

## 9. CREATE TABLE AS

Create a new table with data in a single atomic snapshot.

In [ ]:
from ducklake_pandas import create_table_as_ducklake

summary = (
    read_ducklake(CATALOG, "events")
    .groupby("region")
    .size()
    .reset_index(name="cnt")
)
create_table_as_ducklake(summary, CATALOG, "region_summary")
read_ducklake(CATALOG, "region_summary")

## 10. Views

In [ ]:
from ducklake_pandas import create_ducklake_view, list_views

create_ducklake_view(
    CATALOG, "active_users",
    "SELECT id, name FROM users WHERE region = 'us'",
)
print("Created view 'active_users'")

# Replace view
create_ducklake_view(
    CATALOG, "active_users",
    "SELECT id, name, contact FROM users WHERE region = 'us'",
    or_replace=True,
)
print("Replaced view 'active_users'")

print(f"Views: {list_views(CATALOG)}")

## 10.5. Catalog Migration

Older DuckLake catalogs (v0.3, v0.4) can be brought up to v1.0 in place via `migrate_catalog`. The function is idempotent. Migration is opt-in — reads against older catalogs work without it, but v1.0-only writer features need the upgrade.


In [ ]:
import shutil
from ducklake_pandas import migrate_catalog

# Migrate a copy so the main CATALOG stays at its current version.
side_catalog = os.path.join(WORKDIR, "copy.ducklake")
shutil.copyfile(CATALOG, side_catalog)

before = DuckLakeCatalog(side_catalog).options()
before_version = before.loc[before["key"] == "version", "value"].iloc[0]
print(f"sibling catalog version before: {before_version}")

new_v = migrate_catalog(side_catalog)
print(f"sibling catalog version after:  {new_v}")
print(f"second call (idempotent): {migrate_catalog(side_catalog)!r}")


## 11. Tags

In [ ]:
from ducklake_pandas import (
    set_ducklake_table_tag,
    set_ducklake_column_tag,
    delete_ducklake_table_tag,
)

set_ducklake_table_tag(CATALOG, "users", "owner", "analytics-team")
set_ducklake_table_tag(CATALOG, "users", "pii", "true")
# DuckDB only round-trips `comment` for columns; table tags accept any key on 1.5+.
set_ducklake_column_tag(CATALOG, "users", "contact", "comment", "PII: email")
print("Set tags on users table and contact column")

delete_ducklake_table_tag(CATALOG, "users", "pii")
print("Deleted 'pii' tag")

## 11.5. Macros

DuckLake macros are user-defined SQL functions registered in the catalog. With DuckDB 1.5+ the `duckdb` dialect macros can be called from DuckDB SQL.


In [ ]:
from ducklake_pandas import create_ducklake_macro, drop_ducklake_macro

create_ducklake_macro(
    CATALOG, "add_one", "a + 1",
    parameters=[{"name": "a", "type": "integer"}],
)
print("Macros:", catalog.list_macros()["macro_name"].tolist())

con = duckdb.connect()
con.execute("LOAD ducklake")
con.execute(f"ATTACH 'ducklake:sqlite:{CATALOG}' AS lake (DATA_PATH '{DATAPATH}')")
print("add_one(41) =", con.execute("SELECT lake.add_one(41)").fetchone()[0])
con.close()

drop_ducklake_macro(CATALOG, "add_one")



## 12. Schema and Table Management

In [ ]:
from ducklake_pandas import (
    create_ducklake_schema,
    drop_ducklake_schema,
    rename_ducklake_table,
    drop_ducklake_table,
)

create_ducklake_schema(CATALOG, "staging")
create_ducklake_table(CATALOG, "raw_data", {"x": "INTEGER"}, schema="staging")
print("Created staging.raw_data")

rename_ducklake_table(CATALOG, "raw_data", "raw_events", schema="staging")
print("Renamed staging.raw_data -> staging.raw_events")

drop_ducklake_table(CATALOG, "raw_events", schema="staging")
drop_ducklake_schema(CATALOG, "staging")
print("Dropped staging schema and table")

## 12.5. Streaming Ingestion

`DuckLakeStreamWriter` buffers micro-batches and flushes when the threshold is hit. Exiting the context with an exception drops the unflushed buffer; already-flushed batches remain visible.


In [ ]:
from ducklake_pandas import DuckLakeStreamWriter

with DuckLakeStreamWriter(
    CATALOG, "events", flush_threshold=2, compact_on_close=False,
) as writer:
    writer.append(pd.DataFrame({
        "ts":      pd.to_datetime(["2025-04-01 09:00:00", "2025-04-01 09:05:00"]),
        "user_id": [1, 2],
        "action":  ["login", "login"],
        "region":  ["us", "eu"],
    }))  # auto-flushes (threshold=2)
    writer.append(pd.DataFrame({
        "ts":      pd.to_datetime(["2025-04-01 09:10:00"]),
        "user_id": [1],
        "action":  ["click"],
        "region":  ["us"],
    }))  # buffered → flushes on close

print(f"Wrote {writer.total_rows} rows in {writer.flush_count} flushes")


## 12.6. Registering External Parquet Files

`add_files_ducklake` registers an existing Parquet file into the catalog without copying it.


In [ ]:
import os
from ducklake_pandas import add_files_ducklake

ext_path = os.path.join(WORKDIR, "external.parquet")
pd.DataFrame({
    "ts":      pd.to_datetime(["2025-05-01 10:00:00", "2025-05-01 10:05:00"]),
    "user_id": [1, 2],
    "action":  ["signup", "signup"],
    "region":  ["us", "eu"],
}).to_parquet(ext_path)

added = add_files_ducklake(CATALOG, "events", [ext_path])
print(f"Registered {added} external file(s); events now has {len(read_ducklake(CATALOG, 'events'))} rows")


## 12.7. Compaction

`merge_adjacent_files_ducklake` (DuckLake v1.0+) merges adjacent small files. `cleanup_old_files_ducklake` physically deletes the retired source files. `rewrite_data_files_ducklake` is the heavier alternative.


In [ ]:
from datetime import datetime, timedelta, timezone
from ducklake_pandas import (
    merge_adjacent_files_ducklake,
    cleanup_old_files_ducklake,
)

n_before = len(DuckLakeCatalog(CATALOG).list_files("events"))
merge_adjacent_files_ducklake(CATALOG, "events", min_file_size=1, max_file_size=10_000_000)
n_after = len(DuckLakeCatalog(CATALOG).list_files("events"))
print(f"events: {n_before} files -> {n_after} files")

future = datetime.now(timezone.utc) + timedelta(days=1)
removed = cleanup_old_files_ducklake(CATALOG, older_than=future)
print(f"Cleaned up {len(removed)} retired data files")


## 13. Catalog Maintenance

Clean up old snapshots and orphaned files.

In [ ]:
from ducklake_pandas import expire_snapshots, vacuum_ducklake

catalog = DuckLakeCatalog(CATALOG)
print(f"Before: {len(catalog.snapshots())} snapshots")

expired = expire_snapshots(CATALOG, keep_last_n=3)
vacuumed = vacuum_ducklake(CATALOG)
print(f"Expired {expired} snapshots, vacuumed {vacuumed} orphan files")

catalog = DuckLakeCatalog(CATALOG)
print(f"After:  {len(catalog.snapshots())} snapshots")

## 14. Final Verification

In [ ]:
print("=== ducklake_pandas ===")
for tbl in ["users", "events", "tiny", "region_summary"]:
    n = len(read_ducklake(CATALOG, tbl))
    print(f"  {tbl:20s} → {n} rows")

## Cleanup

In [ ]:
shutil.rmtree(WORKDIR)
print(f"Cleaned up {WORKDIR}")

---

## Summary

This tutorial covered:

| Feature | Functions Used |
|---------|---------------|
| **Read** | `read_ducklake(predicate=lambda d: ..., columns=[...])` |
| **Write** | `write_ducklake(mode="append"/"overwrite")` |
| **Delete** | `delete_ducklake(predicate=lambda d: ...)` |
| **Update** | `update_ducklake(updates={...}, predicate=lambda d: ...)` |
| **Merge** | `merge_ducklake(source_df, on="key")` |
| **Time travel** | `read_ducklake(snapshot_version=N)` |
| **Schema evolution** | `alter_ducklake_add_column`, `_rename_column`, `_drop_column`, `_set_type` |
| **Partitioning** | `alter_ducklake_set_partitioned_by` |
| **Data inlining** | `write_ducklake(data_inlining_row_limit=100)` |
| **CREATE TABLE AS** | `create_table_as_ducklake(df, ...)` |
| **Views** | `create_ducklake_view`, `list_views` |
| **Tags** | `set_ducklake_table_tag`, `set_ducklake_column_tag` |
| **Maintenance** | `expire_snapshots`, `vacuum_ducklake` |
| **Catalog inspect** | `DuckLakeCatalog.snapshots()`, `.list_tables()`, `.list_files()` |

> **Key difference from Polars:** Pandas uses `lambda d: d["col"] == value` for predicates instead of `pl.col("col") == value`, and column types are passed as DuckDB type strings (`"VARCHAR"`, `"BIGINT"`) instead of Polars types (`pl.String`, `pl.Int64`).